In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, skew

# Import clean modular structure from our src folder
from src.data_processing import load_financial_inclusion_data

# Ensure visual output directories exist
os.makedirs("reports/figures", exist_ok=True)

# 1. CLEAN INGESTION CALL 
df = load_financial_inclusion_data("data/processed/ethiopia_fi_unified_data_enriched.csv")

# ==========================================
# 2. ADVANCED QUANTITATIVE ANALYSIS & DATA PROFILE
# ==========================================
print("=== DEEP QUANTITATIVE DATA METRICS ===")

# Calculate strict null allocation percentages across critical features
total_records = len(df)
null_counts = df.isnull().sum()
print("\n--- Structural Completeness & Missing Patterns ---")
for col in df.columns:
    missing_pct = (null_counts[col] / total_records) * 100
    print(f"• Column '{col}': {missing_pct:.2f}% missing values.")

# Structural Volume Breakdown
print("\n--- Volumetric Categorical Balances ---")
print(df.groupby(['record_type']).size().to_string())

print("\n--- Pillar Distribution (Observation Scope) ---")
print(df.groupby(['pillar']).size().to_string())

# Calculate Growth Rates (Compound Annual Variance) for Access vs. Usage
df_obs = df[df['record_type'] == 'observation'].copy()
df_access = df_obs[df_obs['indicator_code'] == 'ACC_OWNERSHIP'].sort_values('year')
df_usage = df_obs[df_obs['indicator_code'] == 'USG_DIGITAL_PAYMENT'].sort_values('year')

# Quantify total growth absolute changes
access_delta = (df_access['value_numeric'].iloc[-1] - df_access['value_numeric'].iloc[0]) * 100
usage_delta = (df_usage['value_numeric'].iloc[-1] - df_usage['value_numeric'].iloc[0]) * 100
print(f"\n--- Trajectory Metrics (2011-2024) ---")
print(f"• Account Ownership (Access) absolute growth: +{access_delta:.2f}% percentage points.")
print(f"• Digital Payment Usage absolute growth: +{usage_delta:.2f}% percentage points.")

# ==========================================
# 3. GRAPHICAL EVALUATIONS & EXPLICIT CORRELATIONS
# ==========================================

# VISUALIZATION 1: Temporal Coverage Matrix
plt.figure(figsize=(10, 5))
coverage_matrix = df_obs.pivot_table(
    index='indicator_code', 
    columns='year', 
    values='value_numeric', 
    aggfunc='count'
).fillna(0)

sns.heatmap(coverage_matrix, cmap="Blues", annot=True, cbar=False, linewidths=1, linecolor='lightgray')
plt.title("Indicator Data Presence Over Time (Observations Matrix)", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Indicator Code")
plt.xlabel("Year")
plt.tight_layout()
plt.savefig("reports/figures/eda_temporal_coverage.png")
plt.show()

# VISUALIZATION 2: Access & Usage Progress Trajectory
plt.figure(figsize=(10, 5))
plt.plot(df_access['year'], df_access['value_numeric'] * 100, marker='o', linewidth=2.5, color='#2b5c8f', label="Access (Account Ownership)")
plt.plot(df_usage['year'], df_usage['value_numeric'] * 100, marker='s', linewidth=2.5, color='#117a65', label="Usage (Digital Payments)")

for x, y in zip(df_access['year'], df_access['value_numeric']):
    plt.annotate(f"{y*100:.1f}%", (x, y*100), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color='#2b5c8f')
for x, y in zip(df_usage['year'], df_usage['value_numeric']):
    plt.annotate(f"{y*100:.1f}%", (x, y*100), textcoords="offset points", xytext=(0,-15), ha='center', fontweight='bold', color='#117a65')

plt.title("Ethiopia Financial Inclusion: Access vs. Usage (2011–2024)", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Inclusion Rate (%)")
plt.xlabel("Survey Year")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("reports/figures/eda_access_vs_usage.png")
plt.show()

# VISUALIZATION 3: Event Overlaid Trajectory
plt.figure(figsize=(11, 5.5))
plt.plot(df_access['year'], df_access['value_numeric'] * 100, marker='o', linewidth=2, color='#2b5c8f', label="Account Ownership Rate")

events_list = [
    (2021.36, "Telebirr Launch", "red"),
    (2022.66, "Safaricom Entry", "orange"),
    (2023.62, "M-Pesa Launch", "green")
]
for year_point, label, color in events_list:
    plt.axvline(x=year_point, color=color, linestyle='--', alpha=0.8, linewidth=1.5)
    plt.text(year_point + 0.1, 15, label, rotation=90, verticalalignment='bottom', fontweight='bold', color=color, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.title("Ethiopia: Account Ownership vs. Market Intervention Events", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Inclusion Rate (%)")
plt.xlabel("Year Line")
plt.xlim(2010, 2026)
plt.ylim(0, 100)
plt.grid(True, alpha=0.2)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("reports/figures/eda_event_overlays.png")
plt.show()

# VISUALIZATION 4: Infrastructure vs. Inclusion Quantitative Correlation
# Interpolating matching alignment matrices to calculate Pearson's stat coefficient
years_corr = [2017, 2021, 2024]
infra_4g = [15.0, 44.0, 58.0]
access_rates = [35.0, 46.0, 49.0]

# Calculate explicit Pearson correlation metric mathematically
r_coeff, p_value = pearsonr(infra_4g, access_rates)
infra_skew = skew(infra_4g)

print("\n--- Statistical Correlation Verification ---")
print(f"• Exact Calculated Pearson Correlation Coefficient (r): {r_coeff:.4f}")
print(f"• P-value significance index: {p_value:.4f}")
print(f"• Infrastructure Data Feature Skewness distribution: {infra_skew:.4f}")

plt.figure(figsize=(6, 5))
sns.regplot(x=infra_4g, y=access_rates, color='#117a65', scatter_kws={'s':100})
plt.title(f"4G Network Coverage vs. Account Ownership\n(Pearson r = {r_coeff:.3f})", fontsize=11, fontweight='bold', pad=12)
plt.xlabel("4G Population Coverage (%)")
plt.ylabel("Findex Account Ownership (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reports/figures/eda_infra_correlation.png")
plt.show()

print("\nAll exploratory analysis plots written to 'reports/figures/' folder.")